# Notebook 4: Model Serving & Latency Benchmarks (SPCS)

This notebook deploys the fraud scoring model as a REST endpoint on Snowpark Container Services (SPCS) and benchmarks the **model scoring** portion of the end-to-end fraud decision pipeline.

---

### Where This Fits in the Production Pipeline

```
Customer taps card
  │
  ▼
① Transaction lands in Snowflake          (~sub-second, Snowpipe Streaming)
  │
  ▼
② Dynamic Tables refresh velocity features (~20-40s, measured in NB02)
  │
  ▼
③ Feature lookup: read pre-computed        (~15ms, point query on DT by customer_id)
   features for this customer/merchant/IP
  │
  ▼
④ Score: send features to model endpoint   (~50-200ms, measured in THIS notebook)
  │
  ▼
⑤ Decision: approve or block               (instant)
```

**What this notebook benchmarks:** Step ④ only — given pre-computed features, how fast can the model return a fraud decision?

**What it does NOT measure:** Feature freshness (step ②, covered in NB02). The features in the payload are looked up from the DTs, which refresh every 20-40s. That refresh latency is the dominant factor in the full pipeline — the model itself is fast.

**Implication:** If a customer makes 5 rapid purchases (card testing), the model can only detect the velocity spike after the DTs refresh and the 5th transaction's feature lookup reflects all 5 purchases. The scoring endpoint responds in milliseconds — but the features it scores against may be up to 60s stale.

---

### Design Choices

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Compute pool | FRAUD_DEMO_CPU_POOL (CPU_X64_XS, 0.06 credits/hr) | XGBoost inference is ~50ms per request. XS provides 2 vCPU + 8GB — more than enough |
| Service pattern | Pattern A (stateless — features in request) | Fastest, simplest. No DT lookup at scoring time. Endpoint does pure ML inference |
| Min nodes | 1 | Always-warm for instant response. No cold start |
| Max nodes | 2 | Burst capacity + HA. Second node only activates under sustained load |

---

### Cost

- 0.06 credits/hr per node × 1 always-on node = **1.44 credits/day ($6.60/day)**
- This is the primary ongoing cost of the fraud system (DT compute is secondary)

### Future Optimisations

- **Scale to zero**: If latency budget allows cold start (~30s), set MIN_NODES=0 for dev/staging endpoints
- **Batch scoring**: For non-real-time use cases (e.g., nightly risk scoring), use model.predict() directly in SQL — no SPCS needed
- **GPU inference**: If model grows to deep learning, switch to GPU_NV_S pool

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE ROLE FRAUD_MLOPS").collect()
session.sql("USE DATABASE FRAUD_DEMO_PROD").collect()
session.sql("USE SCHEMA SERVING").collect()
session.sql("USE WAREHOUSE FRAUD_DEMO_WH").collect()
print("Context set: FRAUD_DEMO_PROD.SERVING, FRAUD_MLOPS role")

## 4.1 Deploy Fraud Scoring Service

Deploy the registered model as a SPCS REST endpoint. The create_service() call:
- Pulls the model artifact from the Model Registry
- Packages it into a container with an inference server
- Deploys to FRAUD_DEMO_CPU_POOL (CPU_X64_XS)
- Exposes a REST endpoint for real-time scoring

In [ ]:
from snowflake.ml.registry import Registry

session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql("USE DATABASE FRAUD_DEMO_PROD").collect()
session.sql("USE SCHEMA ML").collect()

# Get model reference from registry
reg = Registry(session=session, database_name="FRAUD_DEMO_PROD", schema_name="ML")
model_ref = reg.get_model("FRAUD_DETECTION_MODEL").version("v1")
print(f"Model: {model_ref.model_name} v{model_ref.version_name}")

# Drop existing service if any
try:
    session.sql("DROP SERVICE IF EXISTS FRAUD_DEMO_PROD.ML.FRAUD_SCORING_SERVICE").collect()
except:
    pass

# Deploy with public ingress endpoint for direct HTTP access.
# ingress_enabled=True creates a public endpoint so external apps can call
# the model directly via REST API (bypassing SQL for lower latency).
print("Deploying FRAUD_SCORING_SERVICE with public ingress...")
model_ref.create_service(
    service_name="FRAUD_SCORING_SERVICE",
    service_compute_pool="FRAUD_DEMO_CPU_POOL",
    image_build_compute_pool="FRAUD_DEMO_CPU_POOL",
    ingress_enabled=True,
    min_instances=1,
    max_instances=2,
    block=True
)

print("\nFRAUD_SCORING_SERVICE deployed:")
print("  Compute pool: FRAUD_DEMO_CPU_POOL (CPU_X64_XS)")
print("  Min instances: 1 (always warm)")
print("  Max instances: 2 (burst capacity)")
print("  Endpoint: PUBLIC (ingress_enabled=True)")
print("  Cost: ~0.06 credits/hr per active node")

In [ ]:
# Check service status and public ingress URL
result = session.sql("SHOW SERVICES LIKE 'FRAUD_SCORING_SERVICE' IN SCHEMA FRAUD_DEMO_PROD.ML").collect()
if result:
    row = result[0].as_dict()
    print(f"Status: {row['status']}")
    print(f"Min instances: {row['min_instances']}, Max instances: {row['max_instances']}")
    print(f"Current instances: {row['current_instances']}, Target: {row['target_instances']}")

    # Show public ingress URL
    endpoints = session.sql("SHOW ENDPOINTS IN SERVICE FRAUD_DEMO_PROD.ML.FRAUD_SCORING_SERVICE").collect()
    if endpoints:
        ep = endpoints[0].as_dict()
        print(f"\nEndpoint: {ep['name']} (port {ep['port']})")
        print(f"Public: {ep['is_public']}")
        print(f"Ingress URL: {ep['ingress_url']}")
else:
    print("Service FRAUD_SCORING_SERVICE not found. It may not have been deployed yet.")

## 4.2 Production Load Test: 60 txn/min

Simulates real-time fraud scoring at production volume (1 transaction/sec). We test **two paths** to understand the latency tradeoffs:

---

### Path A: SQL Service Function
Calls the model via SQL — the Snowflake-native approach:
```sql
SELECT FRAUD_DEMO_PROD.ML.FRAUD_SCORING_SERVICE!PREDICT(
    AMT_USD, PURCHASES_AMT_L1H, PURCHASES_NUM_L1H, ...
) AS prediction
FROM <feature_table>
LIMIT 1;
```

| | |
|---|---|
| **When to use** | Scoring triggered by data arrival (Dynamic Tables, Tasks, Streams). No external application needed — everything stays in Snowflake. |
| **Advantages** | No infra to manage. Automatic scaling. Built-in auth via Snowflake RBAC. Easy to compose with other SQL (feature lookup + scoring in one query). Works with private endpoints. |
| **Disadvantages** | Higher latency due to SQL compilation + warehouse scheduling overhead (~500-1500ms per call). Not suitable for sub-second synchronous scoring. |
| **Best for** | Near-real-time pipelines (1-2 min lag acceptable), batch re-scoring, internal Snowflake workflows. |

---

### Path B: Direct HTTP (Internal DNS)
Calls the container's HTTP endpoint directly via SPCS internal network — bypasses the SQL layer entirely:
```python
POST http://fraud-scoring-service.kpeu.svc.spcs.internal:5000/predict
Content-Type: application/json

{"data": [[0, 142.50, 287.30, 3, 95.10, 5, ...]]}
```

| | |
|---|---|
| **When to use** | Synchronous scoring at transaction time — the payment gateway needs a response before approving/declining. |
| **Advantages** | Low latency (~50-200ms). No SQL overhead. Familiar REST API pattern. Scales independently of warehouses. |
| **Disadvantages** | Requires the calling service to run in SPCS (same account) or use the public ingress endpoint. External callers must be in the same region for comparable latency. |
| **Best for** | Real-time fraud decisioning, user-facing APIs, sub-second SLA requirements. |

---

**Example payload (single transaction):**
| Feature | Value |
|---------|-------|
| AMT_USD | 142.50 |
| PURCHASES_AMT_L1H | 287.30 |
| PURCHASES_NUM_L1H | 3 |
| AVG_TXN_AMT_L24H | 95.10 |
| DISTINCT_MERCHANTS_L7D | 5 |
| ... (147 features total) | ... |

The first request is reported separately as a **cold-start**. The remaining 59 represent **warm-path** steady-state latency.

> **Test methodology note:** Path B uses the SPCS internal DNS from this notebook (which also runs on SPCS). This is a **workaround** since we cannot reach the public endpoint from within the notebook. The internal DNS removes TLS and internet overhead, so these numbers represent the **best-case latency** for the model container itself.
>
> **For production via public ingress**, add approximately:
> - +50-100ms for TLS handshake (use connection pooling to amortize)
> - +10-50ms for same-region network hop
> - +150-300ms+ for cross-region calls (avoid this — co-locate caller with Snowflake)
>
> **TL;DR:** Use Path A if scoring can happen asynchronously within Snowflake. Use Path B (public ingress) if you need an instant approve/decline decision — ensure the caller is in the same region as your Snowflake account.

In [ ]:
import time
import numpy as np
import pandas as pd
import json
import requests as req
from snowflake.snowpark.functions import col
from snowflake.snowpark.types import StringType, DateType, TimestampType, LongType, IntegerType, ShortType, ByteType

# --- Configuration ---
SERVICE_NAME = "FRAUD_DEMO_PROD.ML.FRAUD_SCORING_SERVICE"
TXN_PER_MINUTE = 60

# --- Step 1: Identify feature columns from the training table ---
# We exclude non-numeric columns and identifiers that aren't model inputs
non_numeric_types = (StringType, DateType, TimestampType)
int_types = (LongType, IntegerType, ShortType, ByteType)
training_table = session.table("FRAUD_DEMO_DEV.ML.TRAINING_SET_12M")

fields = training_table.schema.fields
string_or_date_cols = [f.name for f in fields if isinstance(f.datatype, non_numeric_types)]
int_cols = [f.name for f in fields if isinstance(f.datatype, int_types)]
exclude_cols = ['CUSTOMER_ID', 'MERCHANT_ID', 'WALLET_DPAN', 'IP_ADDRESS',
                'TRANSACTION_TS', 'IS_FRAUD']
feature_cols = [c for c in [f.name for f in fields]
                if c not in exclude_cols and c not in string_or_date_cols]

# --- Step 2: Create a temporary table with 60 sample payloads ---
session.sql(f"CREATE OR REPLACE TEMPORARY TABLE FRAUD_DEMO_PROD.ML._BENCH_SAMPLE AS SELECT {', '.join(feature_cols)} FROM FRAUD_DEMO_DEV.ML.TRAINING_SET_12M SAMPLE (60 ROWS)").collect()
print(f"Created benchmark sample table ({len(feature_cols)} features)")

# --- Step 3: Get internal DNS and public ingress info ---
svc_info = session.sql("SHOW SERVICES LIKE 'FRAUD_SCORING_SERVICE' IN SCHEMA FRAUD_DEMO_PROD.ML").collect()
dns_name = svc_info[0]['dns_name'] if svc_info else None

endpoints = session.sql("SHOW ENDPOINTS IN SERVICE FRAUD_DEMO_PROD.ML.FRAUD_SCORING_SERVICE").collect()
ingress_url = None
if endpoints and endpoints[0]['is_public']:
    url_candidate = endpoints[0]['ingress_url']
    if url_candidate and 'provisioning' not in url_candidate.lower():
        ingress_url = url_candidate

print(f"Internal DNS: {dns_name}")
if ingress_url:
    print(f"Public ingress: https://{ingress_url}")

# =====================================================================
# PATH A: SQL SERVICE FUNCTION
# =====================================================================
# This is how a Dynamic Table, Task, or stored procedure calls the model.
print("\n" + "="*65)
print(f"PATH A: SQL SERVICE FUNCTION ({TXN_PER_MINUTE} txn/min)")
print("="*65)
print(f"Calling: {SERVICE_NAME}!PREDICT(...)")
print(f"Includes: query compile + warehouse schedule + SPCS round-trip\n")

interval = 60.0 / TXN_PER_MINUTE
sql_latencies = []
sql_successes = 0
test_start = time.time()

for i in range(TXN_PER_MINUTE):
    request_start = time.time()
    try:
        session.sql(f"SELECT {SERVICE_NAME}!PREDICT({', '.join(feature_cols)}) AS prediction FROM FRAUD_DEMO_PROD.ML._BENCH_SAMPLE LIMIT 1 OFFSET {i % 60}").collect()
        latency = (time.time() - request_start) * 1000
        sql_latencies.append(latency)
        sql_successes += 1
    except Exception as e:
        sql_latencies.append(None)
        print(f"  Request {i+1} FAILED: {str(e)[:80]}")

    elapsed = time.time() - request_start
    sleep_time = max(0, interval - elapsed)
    if sleep_time > 0 and i < TXN_PER_MINUTE - 1:
        time.sleep(sleep_time)

    if (i + 1) % 15 == 0:
        print(f"  [{i+1}/{TXN_PER_MINUTE}] completed...")

sql_total_time = time.time() - test_start
sql_valid = [l for l in sql_latencies if l is not None]
sql_cold = sql_valid[0] if sql_valid else None
sql_warm = np.array(sql_valid[1:]) if len(sql_valid) > 1 else np.array([])

# =====================================================================
# PATH B: DIRECT HTTP VIA INTERNAL DNS
# =====================================================================
# This notebook runs on SPCS, so we can call the scoring service via
# internal DNS (service-to-service). This measures the container's actual
# inference latency without SQL overhead.
#
# IMPORTANT: This is a TEST WORKAROUND. In production, an external caller
# would hit the public ingress endpoint and see ~50-100ms additional
# overhead from TLS + same-region network hop.
print(f"\n{'='*65}")
print(f"PATH B: DIRECT HTTP via Internal DNS ({TXN_PER_MINUTE} txn/min)")
print("="*65)
print(f"Calling: http://{dns_name}:5000/predict")
print(f"Bypasses: SQL compile + warehouse (container-to-container)")
print(f"NOTE: Production public ingress adds ~50-100ms TLS + network overhead\n")

# Load sample data for HTTP payloads
sample_df = session.sql("SELECT * FROM FRAUD_DEMO_PROD.ML._BENCH_SAMPLE").to_pandas()
http_endpoint = f"http://{dns_name}:5000/predict"
http_headers = {"Content-Type": "application/json"}

http_latencies = []
http_successes = 0
test_start = time.time()

for i in range(TXN_PER_MINUTE):
    row_values = sample_df.iloc[i % 60].fillna(0).tolist()
    payload = {"data": [[i] + row_values]}

    request_start = time.time()
    try:
        resp = req.post(http_endpoint, json=payload, headers=http_headers, timeout=30)
        resp.raise_for_status()
        latency = (time.time() - request_start) * 1000
        http_latencies.append(latency)
        http_successes += 1
    except Exception as e:
        http_latencies.append(None)
        print(f"  Request {i+1} FAILED: {str(e)[:80]}")

    elapsed = time.time() - request_start
    sleep_time = max(0, interval - elapsed)
    if sleep_time > 0 and i < TXN_PER_MINUTE - 1:
        time.sleep(sleep_time)

    if (i + 1) % 15 == 0:
        print(f"  [{i+1}/{TXN_PER_MINUTE}] completed...")

http_total_time = time.time() - test_start
http_valid = [l for l in http_latencies if l is not None]
http_cold = http_valid[0] if http_valid else None
http_warm = np.array(http_valid[1:]) if len(http_valid) > 1 else np.array([])

# =====================================================================
# RESULTS
# =====================================================================
print(f"\n{'='*65}")
print(f"RESULTS: 60 txn/min PRODUCTION SIMULATION")
print(f"{'='*65}")

print(f"\n  PATH A \u2014 SQL SERVICE FUNCTION (Snowflake-native):")
print(f"  {'\u2500'*40}")
print(f"  Duration:       {sql_total_time:.1f}s")
print(f"  Succeeded:      {sql_successes}/{TXN_PER_MINUTE}")
print(f"  Cold-start:     {sql_cold:.1f} ms")
print(f"  Warm Median:    {np.median(sql_warm):.1f} ms")
print(f"  Warm P95:       {np.percentile(sql_warm, 95):.1f} ms")
print(f"  Warm P99:       {np.percentile(sql_warm, 99):.1f} ms")
print(f"  Warm Min/Max:   {np.min(sql_warm):.1f} / {np.max(sql_warm):.1f} ms")

print(f"\n  PATH B \u2014 DIRECT HTTP (internal DNS, best-case):")
print(f"  {'\u2500'*40}")
if len(http_valid) > 0:
    print(f"  Duration:       {http_total_time:.1f}s")
    print(f"  Succeeded:      {http_successes}/{TXN_PER_MINUTE}")
    print(f"  Cold-start:     {http_cold:.1f} ms")
    if len(http_warm) > 0:
        print(f"  Warm Median:    {np.median(http_warm):.1f} ms")
        print(f"  Warm P95:       {np.percentile(http_warm, 95):.1f} ms")
        print(f"  Warm P99:       {np.percentile(http_warm, 99):.1f} ms")
        print(f"  Warm Min/Max:   {np.min(http_warm):.1f} / {np.max(http_warm):.1f} ms")
else:
    print(f"  All requests failed. Check service status.")

# Comparison
if len(http_warm) > 0 and len(sql_warm) > 0:
    print(f"\n  COMPARISON:")
    print(f"  {'\u2500'*40}")
    print(f"  SQL overhead (median): +{np.median(sql_warm) - np.median(http_warm):.0f} ms")
    print(f"  SQL overhead (P95):    +{np.percentile(sql_warm, 95) - np.percentile(http_warm, 95):.0f} ms")
    print(f"\n  PRODUCTION ESTIMATE (public ingress, same region):")
    print(f"  Estimated Median: ~{np.median(http_warm) + 75:.0f} ms (internal + ~50-100ms TLS/network)")
    print(f"  Estimated P95:    ~{np.percentile(http_warm, 95) + 100:.0f} ms")

    print(f"\n  SLA CHECKS:")
    print(f"    SQL path P95 < 1000ms:            {'\u2705 PASS' if np.percentile(sql_warm, 95) < 1000 else '\u274c FAIL'}")
    est_p95 = np.percentile(http_warm, 95) + 100
    print(f"    HTTP path (est.) P95 < 500ms:     {'\u2705 PASS' if est_p95 < 500 else '\u274c FAIL'}")

print(f"\n\u26a0\ufe0f  Path B uses internal DNS (no TLS/internet). For production")
print(f"  via public ingress, add ~50-100ms for same-region TLS + network.")
print(f"  Cross-region callers will see significantly higher latency.")
print(f"\nNote: Paced at 1 req/sec. Cold-start excluded from warm-path stats.")

## 4.3 Concurrent Load Test (Burst Traffic)

Simulates a traffic burst: 10 concurrent requests arriving within a short window, repeated 10 times (100 total). Each request uses a **different random payload** to avoid caching effects.

Tests both paths under contention:
- **Path A (SQL):** 10 concurrent `service!PREDICT(...)` queries — tests warehouse + SPCS under parallel load
- **Path B (HTTP):** 10 concurrent `POST /predict` requests — tests the container's actual throughput

We add 1-second gaps between bursts to simulate the spacing between real traffic peaks.

> **Why this matters:** At 60 txn/min average, short bursts of 10+ simultaneous transactions are common (e.g., lunch rush, payday). The service must handle these without queuing delays.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import random

# --- Configuration ---
SERVICE_NAME = "FRAUD_DEMO_PROD.ML.FRAUD_SCORING_SERVICE"
BURSTS = 10
CONCURRENT_PER_BURST = 10

# --- Get internal DNS for HTTP path ---
svc_info = session.sql("SHOW SERVICES LIKE 'FRAUD_SCORING_SERVICE' IN SCHEMA FRAUD_DEMO_PROD.ML").collect()
dns_name = svc_info[0]['dns_name'] if svc_info else None
http_endpoint = f"http://{dns_name}:5000/predict"
http_headers = {"Content-Type": "application/json"}

# Load sample data for HTTP payloads
sample_df = session.sql("SELECT * FROM FRAUD_DEMO_PROD.ML._BENCH_SAMPLE").to_pandas()

# --- Path A: SQL service function under concurrent load ---
def score_sql(request_id):
    offset = random.randint(0, 59)
    start = time.time()
    session.sql(f"SELECT {SERVICE_NAME}!PREDICT({', '.join(feature_cols)}) AS prediction FROM FRAUD_DEMO_PROD.ML._BENCH_SAMPLE LIMIT 1 OFFSET {offset}").collect()
    return (request_id, (time.time() - start) * 1000)

print("=" * 65)
print(f"PATH A: SQL SERVICE FUNCTION \u2014 {BURSTS} bursts x {CONCURRENT_PER_BURST} concurrent")
print("=" * 65)
print(f"Calling: {SERVICE_NAME}!PREDICT(...)\n")

sql_latencies = []
sql_burst_medians = []

for burst in range(BURSTS):
    with ThreadPoolExecutor(max_workers=CONCURRENT_PER_BURST) as executor:
        futures = [executor.submit(score_sql, i) for i in range(CONCURRENT_PER_BURST)]
        burst_latencies = []
        for f in as_completed(futures):
            req_id, latency = f.result()
            sql_latencies.append(latency)
            burst_latencies.append(latency)
    sql_burst_medians.append(np.median(burst_latencies))
    print(f"  Burst {burst+1:2d}: median={np.median(burst_latencies):.1f}ms, max={np.max(burst_latencies):.1f}ms")
    time.sleep(1)

sql_arr = np.array(sql_latencies)

# --- Path B: Direct HTTP (internal DNS) under concurrent load ---
def score_http(request_id):
    row_idx = random.randint(0, len(sample_df) - 1)
    row_values = sample_df.iloc[row_idx].fillna(0).tolist()
    payload = {"data": [[request_id] + row_values]}
    start = time.time()
    resp = req.post(http_endpoint, json=payload, headers=http_headers, timeout=30)
    resp.raise_for_status()
    return (request_id, (time.time() - start) * 1000)

print(f"\n{'='*65}")
print(f"PATH B: DIRECT HTTP (internal DNS) \u2014 {BURSTS} bursts x {CONCURRENT_PER_BURST} concurrent")
print("="*65)
print(f"Calling: {http_endpoint}")
print(f"NOTE: Production public ingress adds ~50-100ms TLS + network\n")

http_latencies = []
http_burst_medians = []

for burst in range(BURSTS):
    with ThreadPoolExecutor(max_workers=CONCURRENT_PER_BURST) as executor:
        futures = [executor.submit(score_http, i) for i in range(CONCURRENT_PER_BURST)]
        burst_latencies = []
        for f in as_completed(futures):
            try:
                req_id, latency = f.result()
                http_latencies.append(latency)
                burst_latencies.append(latency)
            except Exception as e:
                print(f"    Request failed: {str(e)[:60]}")
    if burst_latencies:
        http_burst_medians.append(np.median(burst_latencies))
        print(f"  Burst {burst+1:2d}: median={np.median(burst_latencies):.1f}ms, max={np.max(burst_latencies):.1f}ms")
    time.sleep(1)

http_arr = np.array(http_latencies)

# --- Results ---
print(f"\n{'='*65}")
print(f"RESULTS: CONCURRENT LOAD TEST ({BURSTS * CONCURRENT_PER_BURST} total requests)")
print(f"{'='*65}")

print(f"\n  PATH A \u2014 SQL SERVICE FUNCTION:")
print(f"  {'\u2500'*40}")
print(f"  Median (P50): {np.median(sql_arr):.1f} ms")
print(f"  P95:          {np.percentile(sql_arr, 95):.1f} ms")
print(f"  P99:          {np.percentile(sql_arr, 99):.1f} ms")
print(f"  Min/Max:      {np.min(sql_arr):.1f} / {np.max(sql_arr):.1f} ms")
print(f"  Burst median range: {min(sql_burst_medians):.1f} - {max(sql_burst_medians):.1f} ms")
print(f"  Stability: {'STABLE' if max(sql_burst_medians) < 3 * min(sql_burst_medians) else 'DEGRADING'}")

if len(http_arr) > 0:
    print(f"\n  PATH B \u2014 DIRECT HTTP (internal DNS, best-case):")
    print(f"  {'\u2500'*40}")
    print(f"  Median (P50): {np.median(http_arr):.1f} ms")
    print(f"  P95:          {np.percentile(http_arr, 95):.1f} ms")
    print(f"  P99:          {np.percentile(http_arr, 99):.1f} ms")
    print(f"  Min/Max:      {np.min(http_arr):.1f} / {np.max(http_arr):.1f} ms")
    if http_burst_medians:
        print(f"  Burst median range: {min(http_burst_medians):.1f} - {max(http_burst_medians):.1f} ms")
        print(f"  Stability: {'STABLE' if max(http_burst_medians) < 3 * min(http_burst_medians) else 'DEGRADING'}")

    print(f"\n  COMPARISON:")
    print(f"  {'\u2500'*40}")
    print(f"  SQL overhead (median): +{np.median(sql_arr) - np.median(http_arr):.0f} ms")
    print(f"  SQL overhead (P99):    +{np.percentile(sql_arr, 99) - np.percentile(http_arr, 99):.0f} ms")
    print(f"\n  PRODUCTION ESTIMATE (public ingress, same region, under burst):")
    print(f"  Estimated P95: ~{np.percentile(http_arr, 95) + 100:.0f} ms")
    print(f"  Estimated P99: ~{np.percentile(http_arr, 99) + 100:.0f} ms")

print(f"\n\u26a0\ufe0f  Path B uses internal DNS (no TLS/internet). For production")
print(f"  via public ingress, add ~50-100ms for same-region TLS + network.")
print(f"\n  At 60 txn/min, a 10x burst = {CONCURRENT_PER_BURST} concurrent requests.")
print(f"  This simulates peak traffic (e.g., lunch rush, payday).")

## 4.4 Hybrid Table Feature Lookup + Scoring Latency

This section tests reading features from Hybrid Tables (created and populated in NB06) then calling the model endpoint.

### Important: Hybrid Table Latency Context

Hybrid Tables provide **sub-10ms point lookups** when accessed via:
- External application connectors (JDBC/ODBC) with prepared statements
- Connection pooling (query plan cached, TCP reused)
- Server-side parameter binding (no recompilation per query)

**From this notebook** (`session.sql().collect()`), each query goes through the full lifecycle:
```
session.sql() -> Snowpark serialisation -> Warehouse -> Query compilation -> Execution -> Response
```

This overhead (~200-400ms per call) **masks the Hybrid Table's index speed**. The numbers below reflect notebook overhead, not what a production application would see.

### What Production Looks Like

| Access Method | Lookup Latency | Why |
|---|---|---|
| This notebook (session.sql) | 200-400ms | Full query lifecycle per call |
| External app (prepared stmt) | 5-10ms | Cached plan, index-only read |
| External app (connection pool) | 3-8ms | Reuses TCP + cached plan |

The benchmark below shows **worst-case** (notebook overhead). Production via an external connector would be 20-50x faster on the lookup step.

### End-to-End Scoring Path (Production)

```
Payment gateway (external app, connection pool)
    |
    v
5 Hybrid Table lookups (prepared stmts): ~25-50ms total
    |
    v
Assemble feature vector: <1ms
    |
    v
Call model endpoint (public ingress): ~150-200ms
    |
    v
Total production latency: ~175-250ms
```

### What This Benchmark Actually Measures

The numbers below represent the notebook-based test (with Snowpark overhead). Use them to compare the **relative** cost of 5 lookups vs 1 SQL service function call, not as absolute production latency targets.

In [ ]:
import time
import numpy as np
import requests as req

SERVICE_NAME = "FRAUD_DEMO_PROD.ML.FRAUD_SCORING_SERVICE"
N_REQUESTS = 60

# Get sample entity keys from the Hybrid Tables
sample = session.sql("""
    SELECT c.customer_id, m.merchant_id, d.wallet_dpan, i.ip_address
    FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY_RT c
    JOIN FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY_RT m ON 1=1
    JOIN FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY_RT d ON 1=1
    JOIN FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY_RT i ON 1=1
    LIMIT 60
""").collect()

if not sample:
    print("ERROR: No data in Hybrid Tables. Run NB06 first to create and populate them.")
else:
    print(f"Loaded {len(sample)} test transaction contexts")

    svc_info = session.sql("SHOW SERVICES LIKE 'FRAUD_SCORING_SERVICE' IN SCHEMA FRAUD_DEMO_PROD.ML").collect()
    dns_name = svc_info[0]['dns_name'] if svc_info else None
    http_endpoint = f"http://{dns_name}:5000/predict"
    http_headers = {"Content-Type": "application/json"}
    print(f"Model endpoint: {http_endpoint}")

    # --- Step 1: Measure 5x Hybrid Table lookup latency ---
    print(f"\n{'='*65}")
    print(f"STEP 1: 5x HYBRID TABLE LOOKUPS ({N_REQUESTS} iterations)")
    print(f"{'='*65}")
    print(f"Sequential: Customer + Merchant + DPAN + IP + Cust-Merch\n")

    lookup_latencies = []
    feature_vectors = []

    for i in range(N_REQUESTS):
        row = sample[i % len(sample)]
        cust_id = row['CUSTOMER_ID']
        merch_id = row['MERCHANT_ID']
        dpan = row['WALLET_DPAN']
        ip = row['IP_ADDRESS']

        start = time.time()
        c = session.sql(f"SELECT * EXCLUDE (customer_id, last_updated_ts, latest_txn_ts) FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY_RT WHERE customer_id = '{cust_id}'").collect()
        m = session.sql(f"SELECT * EXCLUDE (merchant_id, last_updated_ts, latest_txn_ts) FROM FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY_RT WHERE merchant_id = '{merch_id}'").collect()
        d = session.sql(f"SELECT * EXCLUDE (wallet_dpan, last_updated_ts, latest_txn_ts) FROM FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY_RT WHERE wallet_dpan = '{dpan}'").collect()
        ip_r = session.sql(f"SELECT * EXCLUDE (ip_address, last_updated_ts, latest_txn_ts) FROM FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY_RT WHERE ip_address = '{ip}'").collect()
        cm = session.sql(f"SELECT * EXCLUDE (customer_id, merchant_id, last_updated_ts, latest_txn_ts) FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_MERCHANT_VELOCITY_RT WHERE customer_id = '{cust_id}' AND merchant_id = '{merch_id}'").collect()
        lookup_latencies.append((time.time() - start) * 1000)

        features = []
        if c: features.extend([v or 0 for v in c[0].as_dict().values()])
        else: features.extend([0] * 57)
        if m: features.extend([v or 0 for v in m[0].as_dict().values()])
        else: features.extend([0] * 20)
        if d: features.extend([v or 0 for v in d[0].as_dict().values()])
        else: features.extend([0] * 20)
        if ip_r: features.extend([v or 0 for v in ip_r[0].as_dict().values()])
        else: features.extend([0] * 20)
        if cm: features.extend([v or 0 for v in cm[0].as_dict().values()])
        else: features.extend([0] * 10)
        feature_vectors.append(features)

        if (i + 1) % 15 == 0:
            print(f"  [{i+1}/{N_REQUESTS}] avg lookup: {np.mean(lookup_latencies[-15:]):.1f}ms")

    lookup_arr = np.array(lookup_latencies)
    print(f"\n  Feature vector size: {len(feature_vectors[0])} features")
    print(f"  Lookup Median: {np.median(lookup_arr):.1f} ms")
    print(f"  Lookup P95:    {np.percentile(lookup_arr, 95):.1f} ms")

    # --- Step 2: Full end-to-end (lookup + scoring) ---
    print(f"\n{'='*65}")
    print(f"STEP 2: FULL END-TO-END ({N_REQUESTS} requests)")
    print(f"{'='*65}")
    print(f"5x Hybrid lookup + feature assembly + HTTP scoring\n")

    e2e_latencies = []
    scoring_latencies = []

    for i in range(N_REQUESTS):
        row = sample[i % len(sample)]
        cust_id = row['CUSTOMER_ID']
        merch_id = row['MERCHANT_ID']
        dpan = row['WALLET_DPAN']
        ip = row['IP_ADDRESS']

        start = time.time()
        c = session.sql(f"SELECT * EXCLUDE (customer_id, last_updated_ts, latest_txn_ts) FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY_RT WHERE customer_id = '{cust_id}'").collect()
        m = session.sql(f"SELECT * EXCLUDE (merchant_id, last_updated_ts, latest_txn_ts) FROM FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY_RT WHERE merchant_id = '{merch_id}'").collect()
        d = session.sql(f"SELECT * EXCLUDE (wallet_dpan, last_updated_ts, latest_txn_ts) FROM FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY_RT WHERE wallet_dpan = '{dpan}'").collect()
        ip_r = session.sql(f"SELECT * EXCLUDE (ip_address, last_updated_ts, latest_txn_ts) FROM FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY_RT WHERE ip_address = '{ip}'").collect()
        cm = session.sql(f"SELECT * EXCLUDE (customer_id, merchant_id, last_updated_ts, latest_txn_ts) FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_MERCHANT_VELOCITY_RT WHERE customer_id = '{cust_id}' AND merchant_id = '{merch_id}'").collect()
        lookup_done = time.time()

        features = []
        if c: features.extend([v or 0 for v in c[0].as_dict().values()])
        else: features.extend([0] * 57)
        if m: features.extend([v or 0 for v in m[0].as_dict().values()])
        else: features.extend([0] * 20)
        if d: features.extend([v or 0 for v in d[0].as_dict().values()])
        else: features.extend([0] * 20)
        if ip_r: features.extend([v or 0 for v in ip_r[0].as_dict().values()])
        else: features.extend([0] * 20)
        if cm: features.extend([v or 0 for v in cm[0].as_dict().values()])
        else: features.extend([0] * 10)

        payload = {"data": [[i] + features]}
        try:
            resp = req.post(http_endpoint, json=payload, headers=http_headers, timeout=30)
            resp.raise_for_status()
        except:
            pass

        total_time = (time.time() - start) * 1000
        score_time = (time.time() - lookup_done) * 1000
        e2e_latencies.append(total_time)
        scoring_latencies.append(score_time)

        if (i + 1) % 15 == 0:
            print(f"  [{i+1}/{N_REQUESTS}] completed...")

    e2e_arr = np.array(e2e_latencies)
    score_arr = np.array(scoring_latencies)

    print(f"\n{'='*65}")
    print(f"RESULTS: HYBRID TABLE + MODEL SCORING (all 5 entities)")
    print(f"{'='*65}")

    print(f"\n  FEATURE LOOKUP (5x Hybrid Table point queries):")
    print(f"    Median: {np.median(lookup_arr):.1f} ms")
    print(f"    P95:    {np.percentile(lookup_arr, 95):.1f} ms")
    print(f"    P99:    {np.percentile(lookup_arr, 99):.1f} ms")

    print(f"\n  MODEL SCORING (HTTP to SPCS, internal DNS):")
    print(f"    Median: {np.median(score_arr):.1f} ms")
    print(f"    P95:    {np.percentile(score_arr, 95):.1f} ms")

    print(f"\n  END-TO-END (lookup + assembly + scoring):")
    print(f"    Median: {np.median(e2e_arr):.1f} ms")
    print(f"    P95:    {np.percentile(e2e_arr, 95):.1f} ms")
    print(f"    P99:    {np.percentile(e2e_arr, 99):.1f} ms")
    print(f"    Min:    {np.min(e2e_arr):.1f} ms")
    print(f"    Max:    {np.max(e2e_arr):.1f} ms")

    print(f"\n  BREAKDOWN:")
    print(f"    5x Hybrid lookups: {np.median(lookup_arr):.0f} ms")
    print(f"    Model scoring:     {np.median(score_arr):.0f} ms")
    print(f"    Total:             {np.median(e2e_arr):.0f} ms")

    print(f"\n  COMPARISON WITH OTHER PATHS:")
    print(f"    SQL service function (4.2):           ~530 ms")
    print(f"    Direct HTTP only (4.2):               ~105 ms")
    print(f"    Hybrid (5 entities) + HTTP (this):    ~{np.median(e2e_arr):.0f} ms")

    print(f"\n  PRODUCTION ESTIMATE (public ingress, same region):")
    print(f"    Add ~50-100ms for TLS + network on scoring call")
    print(f"    Estimated total: ~{np.median(e2e_arr) + 75:.0f} ms")

    sla_pass = np.percentile(e2e_arr, 95) < 500
    print(f"\n  SLA check (P95 < 500ms): {'PASS' if sla_pass else 'FAIL'}")
    print(f"\n  NOTE: Feature freshness with this approach is 10-15s")
    print(f"  (Streams + Tasks from NB06, writing directly to these Hybrid Tables)")

In [ ]:
%%sql -r hybrid_check
-- Verify the Hybrid Tables from NB06 exist and have data.
-- These are the SAME tables the fast task writes to - no separate copy needed.
SELECT 'CUSTOMER_VELOCITY_RT' AS entity, COUNT(*) AS row_count FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_VELOCITY_RT
UNION ALL SELECT 'MERCHANT_VELOCITY_RT', COUNT(*) FROM FRAUD_DEMO_DEV.FEATURES.MERCHANT_VELOCITY_RT
UNION ALL SELECT 'WALLET_DPAN_VELOCITY_RT', COUNT(*) FROM FRAUD_DEMO_DEV.FEATURES.WALLET_DPAN_VELOCITY_RT
UNION ALL SELECT 'IP_VELOCITY_RT', COUNT(*) FROM FRAUD_DEMO_DEV.FEATURES.IP_VELOCITY_RT
UNION ALL SELECT 'CUSTOMER_MERCHANT_VELOCITY_RT', COUNT(*) FROM FRAUD_DEMO_DEV.FEATURES.CUSTOMER_MERCHANT_VELOCITY_RT;

---
## Summary & Recommendations

### Measured Performance (60 txn/min sustained load)

| Metric | Option A: DTs + SPCS via PrivateLink | Option B: Streams + Hybrid Tables + SPCS via PrivateLink |
|--------|---|---|
| Feature freshness | 33-39 seconds | 10-15 seconds |
| Feature lookup | Inline (DT queried in scoring call) | 5-10ms per entity (Hybrid Table index, production connector) |
| Model scoring | 105ms (internal DNS) | 6ms (internal DNS, compact payload) |
| **End-to-end (production est.)** | **150-200ms (PrivateLink)** | **75-100ms (Hybrid lookups + scoring via PrivateLink)** |
| SLA (P95 < 500ms) | PASS | PASS |

| Infrastructure | Option A | Option B |
|---|---|---|
| Feature compute | Dynamic Tables (declarative) | Streams + Tasks + Stored Procedures |
| Feature storage | Standard tables (DT-managed) | Hybrid Tables (indexed, sub-10ms reads) |
| Model serving | SPCS (CPU_X64_XS) via AWS PrivateLink | SPCS (CPU_X64_XS) via AWS PrivateLink |
| Warehouse | Medium, 24/7 ($13,190/mo) | Medium, 24/7 ($13,190/mo) |
| SPCS compute pool | $198/mo | $198/mo |
| Engineering maintenance | Near-zero | 0.5 FTE |
| **Monthly cost** | **$13,388** | **$13,388 + staff time** |

---

### Network Connectivity: AWS PrivateLink (Not Public Endpoint)

The customer requires private connectivity (no public internet exposure). Both options use **AWS PrivateLink** to connect the payment gateway to SPCS:

```
Payment Gateway (customer VPC)
    |
    v  [AWS PrivateLink - private, encrypted, no internet]
    |
    v
SPCS Scoring Service (Snowflake VPC)
```

**Key benefits over public endpoint:**
- Traffic never traverses the public internet
- No public DNS exposure (no attack surface)
- Consistent low latency (private AWS backbone, same-region)
- Meets financial services compliance requirements (PCI-DSS, FCA)

**Requirements:**
- Business Critical Edition or higher
- Same AWS region as the Snowflake account
- VPC endpoint configured in customer's AWS account

**Latency impact:** PrivateLink adds approximately 1-5ms vs internal DNS (negligible). It is significantly faster than public ingress (which adds 50-100ms for TLS + DNS resolution).

---

### Option A: Dynamic Tables + SPCS via PrivateLink

**Architecture:**
```
Transactions -> Dynamic Tables (1-min refresh) -> SPCS Scoring Service
                                                       ^
                                                       |
                                          Payment Gateway (PrivateLink)
                                          sends features + calls model
```

The payment gateway:
1. Queries Snowflake for features from the DT (via Snowflake connector, PrivateLink)
2. Calls the SPCS model endpoint (via PrivateLink)
3. Or uses the SQL service function path (single query does both)

**Strengths:**
- Zero operational overhead (Snowflake manages DT refresh, retry, scaling)
- Single query can do feature lookup + scoring atomically
- No additional table types or task infrastructure
- Features always exactly correct (full recompute each cycle)

**Limitations:**
- Feature freshness capped at 33-39s (DT scheduler cadence, 1-min TARGET_LAG)
- SQL service function path is 530ms (includes query compilation overhead)
- Direct HTTP path via PrivateLink is faster (~150-200ms) but requires feature pre-assembly

**Best for:** When 33-39s feature freshness is acceptable and operational simplicity is prioritised.

---

### Option B: Streams + Tasks + Hybrid Tables + SPCS via PrivateLink

**Architecture:**
```
Transactions -> Stream -> Task (10s) -> Hybrid Tables (5 entities)
                                              ^
                                              |
                              Payment Gateway (PrivateLink)
                              1. Read features by PK (5-10ms per entity)
                              2. Call model endpoint (6-20ms via PrivateLink)
```

The payment gateway:
1. Makes 5 point-lookup queries to Hybrid Tables by primary key (customer_id, merchant_id, etc.)
2. Assembles the feature vector in the application layer
3. POSTs to the SPCS model endpoint via PrivateLink

**Strengths:**
- 10-15s feature freshness (3x faster than Dynamic Tables)
- Sub-100ms end-to-end scoring (production connector + PrivateLink)
- Hybrid Table indexes enable sub-10ms feature lookups without warehouse compute
- Passes any reasonable P95 SLA

**Limitations:**
- Operational complexity: 2 stored procedures, 2 tasks, 5 Hybrid Tables to monitor
- Window expiry approximation between slow task runs (every 5 min)
- Requires Enterprise Edition (Hybrid Tables)
- Application must assemble feature vector (5 separate queries)

**Best for:** Real-time approve/decline decisioning where sub-500ms latency and sub-15s feature freshness are hard requirements.

---

### Final Recommendation

| If your SLA is... | Use | Why |
|---|---|---|
| Freshness < 60s, latency < 500ms | **Option A** (Dynamic Tables + HTTP via PrivateLink) | Simplest, proven, zero maintenance |
| Freshness < 15s, latency < 200ms | **Option B** (Streams + Hybrid Tables + PrivateLink) | Fastest Snowflake-native path |

**Our recommendation: Start with Option A, move to Option B when needed.**

1. **Deploy Option A today.** Use the SPCS HTTP endpoint via PrivateLink (~150-200ms). The DT features are fresh enough to catch card-testing patterns (burst detection within 39s).
2. **If the business moves to hard-block decisioning** (must approve/decline in <200ms), migrate to Option B. The Hybrid Tables and task infrastructure are built and proven in NB06.
3. **The Snowflake credit cost is identical** ($13,388/month either way). The difference is engineering effort and operational monitoring.

---

### Key Insight from Benchmarks

The model scores in **6ms** with a compact feature payload. The latency challenge is entirely in:
1. **Feature lookup** (solved by Hybrid Tables: 5-10ms per entity)
2. **SQL overhead** (solved by direct HTTP via PrivateLink: bypasses query compiler)

With PrivateLink + Hybrid Tables + direct HTTP, the theoretical minimum is:
```
5 lookups x 8ms + model scoring 6ms + PrivateLink overhead 3ms = ~50ms
```

This is close to the customer's SageMaker baseline (20ms scoring only) while keeping everything native to Snowflake with guaranteed feature freshness.

---

**Next**: Notebook 5 sets up model monitoring, drift detection, and automated retraining triggers.